In [ ]:
# ── Cell 1: Install packages ────────────────────────────────────────────
import subprocess, sys, os, signal

result = subprocess.run([sys.executable, '-c', 'import numpy as np; print(np.__version__)'],
                        capture_output=True, text=True)
major = int(result.stdout.strip().split('.')[0])
if major >= 2:
    print(f'NumPy {result.stdout.strip()} — downgrading to <2.0 for wntr compatibility …')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy<2.0',
                    '--quiet', '--upgrade'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install',
                'wntr', 'networkx', 'pandas', 'openpyxl',
                'matplotlib', 'scipy', '--quiet'], check=True)
print('All packages installed.')
if major >= 2:
    print('>>> Restart runtime now (Runtime ▸ Restart session), then run from Cell 2. <<<')
    os.kill(os.getpid(), signal.SIGKILL)


# =====================================================

# ── Cell 2: Imports ─────────────────────────────────────────────────────
import os, sys, math, warnings, random, heapq
from copy import deepcopy
from collections import Counter, defaultdict

warnings.filterwarnings('ignore')
random.seed(42)

import wntr
import networkx as nx
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy.stats import rankdata
from google.colab import files

np.random.seed(42)

INCH_PER_M = 39.3700787402
GAMMA      = 9810.0   # N/m³
G_ACCEL    = 9.81

BASE = '/content/wmugrid_outputs'
for d in [BASE, f'{BASE}/s1', f'{BASE}/s2', f'{BASE}/s3',
          f'{BASE}/s4', f'{BASE}/s5', f'{BASE}/s6']:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130,
                     'axes.spines.top': False,
                     'axes.spines.right': False,
                     'font.family': 'DejaVu Sans'})

print(f'wntr {wntr.__version__} | networkx {nx.__version__} | numpy {np.__version__}')
print(f'Output root: {BASE}')


# =====================================================

# ── Cell 3: Upload ──────────────────────────────────────────────────────
print('Upload your EPANET .inp file ...')
uploaded     = files.upload()
inp_files    = [k for k in uploaded if k.lower().endswith('.inp')]
if not inp_files:
    raise ValueError('No .inp file found.')
INP_PATH     = inp_files[0]
NETWORK_NAME = os.path.splitext(os.path.basename(INP_PATH))[0]
print(f'Loaded: {INP_PATH}  ({len(uploaded[INP_PATH])/1024:.1f} KB)')
print(f'Network: {NETWORK_NAME}')


# =====================================================

# ── Step 1a: Build undirected graph G ───────────────────────────────────
wn         = wntr.network.WaterNetworkModel(INP_PATH)
reservoirs = set(wn.reservoir_name_list)
tanks      = set(wn.tank_name_list)
junctions  = set(wn.junction_name_list)
sources    = reservoirs | tanks

G = nx.Graph()
for n in wn.node_name_list:
    node   = wn.get_node(n)
    bd     = sum(ts.base_value for ts in getattr(node,'demand_timeseries_list',[]))
    coords = getattr(node,'coordinates',(0,0)) or (0,0)
    G.add_node(n,
        kind        = ('reservoir' if n in reservoirs else
                       'tank'      if n in tanks      else 'junction'),
        elevation   = float(getattr(node,'elevation',0.0) or 0.0),
        base_demand = float(bd),
        x=float(coords[0]), y=float(coords[1]))

for p in wn.pipe_name_list:
    lnk = wn.get_link(p)
    u,v = lnk.start_node_name, lnk.end_node_name
    G.add_edge(u,v,
        link_id=p, kind='pipe',
        length=float(lnk.length), diameter=float(lnk.diameter),
        roughness=float(getattr(lnk,'roughness',0.0) or 0.0),
        status=str(lnk.initial_status).replace('LinkStatus.','').upper())

pos = {n:(G.nodes[n]['x'],G.nodes[n]['y']) for n in G.nodes()}
coord_vals = np.array([[d['x'],d['y']] for _,d in G.nodes(data=True)])
if not bool((coord_vals.std(axis=0)>0).all()):
    print('No spatial coords — using spring layout')
    pos = nx.spring_layout(G, seed=42)

pipe_area = {d['link_id']: math.pi*(max(d['diameter'],1e-9)/2)**2
             for _,_,d in G.edges(data=True)}

# DataFrames
df_nodes = pd.DataFrame([
    {'NodeID':n,'Type':d['kind'],'Elevation_m':d['elevation'],
     'BaseDemand_Ls':d['base_demand']*1000,'BaseDemand_m3s':d['base_demand'],
     'Degree':G.degree(n),'IsSource':int(n in sources),'X':d['x'],'Y':d['y']}
    for n,d in G.nodes(data=True)])

df_pipes = pd.DataFrame([
    {'PipeID':d['link_id'],'From':u,'To':v,
     'Length_m':d['length'],'Diameter_m':d['diameter'],
     'Diameter_in':d['diameter']*INCH_PER_M,'Roughness':d['roughness']}
    for u,v,d in G.edges(data=True)
]).sort_values('Diameter_m',ascending=False).reset_index(drop=True)


# =====================================================

# ── Step 1b: Run baseline EPS ───────────────────────────────────────────
print('Running baseline EPS ...')
sim_orig     = wntr.sim.EpanetSimulator(wn)
results_orig = sim_orig.run_sim()

press_ts  = results_orig.node['pressure']
head_ts   = results_orig.node['head']
demand_ts = results_orig.node['demand']
flow_ts   = results_orig.link['flowrate']
flow_abs  = flow_ts.abs()

vel_ts = flow_abs.copy()
for lid in vel_ts.columns:
    vel_ts[lid] = vel_ts[lid] / max(pipe_area.get(lid,1e-9), 1e-9)

hloss_data = {}
for u,v,d in G.edges(data=True):
    lid=d['link_id']
    if u in head_ts.columns and v in head_ts.columns:
        hloss_data[lid]=(head_ts[u]-head_ts[v]).abs()
hloss_ts = pd.DataFrame(hloss_data)

eflux = {lid: float((GAMMA*flow_abs[lid]*hloss_ts[lid]).mean())
         if lid in hloss_ts.columns else 0.0
         for lid in flow_ts.columns}
eflux_series = pd.Series(eflux)

flow_mean  = flow_abs.mean(axis=0)
vel_mean   = vel_ts.mean(axis=0)
vel_max    = vel_ts.max(axis=0)
hloss_mean = hloss_ts.abs().mean(axis=0)
press_mean = press_ts.mean(axis=0)
press_min  = press_ts.min(axis=0)
press_max  = press_ts.max(axis=0)

df_pipes['Q_mean_m3s']    = df_pipes['PipeID'].map(lambda x: float(flow_mean.get(x,np.nan)))
df_pipes['EnergyFlux_W']  = df_pipes['PipeID'].map(lambda x: float(eflux_series.get(x,0.0)))
df_pipes['Headloss_m']    = df_pipes['PipeID'].map(lambda x: float(hloss_mean.get(x,np.nan)))
df_nodes['P_mean_m']      = df_nodes['NodeID'].map(lambda x: float(press_mean.get(x,np.nan)))
df_nodes['P_min_m']       = df_nodes['NodeID'].map(lambda x: float(press_min.get(x,np.nan)))
df_nodes['P_max_m']       = df_nodes['NodeID'].map(lambda x: float(press_max.get(x,np.nan)))


# =====================================================

diams_in = df_pipes['Diameter_in'].dropna().values
print('Pipe diameter percentiles:')
for p in [50,60,70,75,80,85,90]:
    print(f'  P{p:2d}: {np.percentile(diams_in,p):.2f} in')

# ────────── USER PARAMETERS ──────────────────────────────────────────────
MAINS_THRESH_IN = 12.0
SECTOR_MIN      = 5
SECTOR_MAX      = 35
P_MIN           = 14.0   # m
P_MAX           = 100.0  # m
V_MAX           = 3.0    # m/s
N_CANDIDATES    = 300
MAX_SHIFT_OPS   = 3
PRESERVE_CONN   = True
Q_QUANT         = 0.90
E_QUANT         = 0.90
S_QUANT         = 0.85
MAINS_PENALTY   = 1e6
EPS             = 1e-6

V8H_LAMBDA      = 0.60
V8H_BETA        = 0.70
V8H_REV_PEN     = 6.0
V8H_QFLOOR_FRAC = 0.15

MAINS_THRESH_M  = MAINS_THRESH_IN / INCH_PER_M
CMAP10          = plt.cm.tab10
source_list     = sorted(list(sources), key=str)

# Todini ANR baseline
_ri = wntr.metrics.hydraulic.todini_index(
    head_ts, press_ts, demand_ts, flow_ts, wn, P_MIN)
ANR_BASE = float(_ri.mean())

# Dissipated power baseline
DP_BASE = float(np.mean([
    sum(GAMMA*abs(float(flow_abs.loc[t,p]))*abs(float(hloss_ts.loc[t,p]))
        for p in wn.pipe_name_list if p in hloss_ts.columns)
    for t in flow_abs.index]))

# Water age baseline
try:
    _wn2 = wntr.network.WaterNetworkModel(INP_PATH)
    _wn2.options.quality.parameter = 'AGE'
    _r2  = wntr.sim.EpanetSimulator(_wn2).run_sim()
    WA_BASE = float(_r2.node['quality'].reindex(
        columns=list(junctions),fill_value=np.nan).tail(24).values.mean()/3600)
except:
    WA_BASE = None

junc_nodes = [n for n in G.nodes() if G.nodes[n]['kind']=='junction']
jp = press_mean.reindex(junc_nodes)
P_MEAN_BASE = float(jp.mean())
P_MIN_BASE  = float(press_min.reindex(junc_nodes).min())
P_CV_BASE   = float(jp.std()/jp.mean())

print(f'\nBaseline EPS results (Net3 targets in brackets):')
print(f'  Mean pressure : {P_MEAN_BASE:.1f} m   [42.0]')
print(f'  Min  pressure : {P_MIN_BASE:.2f} m  [-0.62]')
print(f'  Pressure CV   : {P_CV_BASE:.3f}     [0.228]')
print(f'  Todini ANR    : {ANR_BASE:.3f}     [0.525]')
print(f'  Dissip. Power : {DP_BASE:,.0f} W  [226,790]')
if WA_BASE: print(f'  Water Age     : {WA_BASE:.1f} h    [14.4]')


# =====================================================

# ── Step 2a: Three-gate mains ──────────────────────────────
n_p = len(df_pipes)
rD  = rankdata(df_pipes['Diameter_m'].values)  / n_p
rQ  = rankdata(df_pipes['Q_mean_m3s'].values)  / n_p
rE  = rankdata(df_pipes['EnergyFlux_W'].values) / n_p

df_pipes['rD']        = rD
df_pipes['rQ']        = rQ
df_pipes['rE']        = rE
df_pipes['MainScore'] = (rD+rQ+rE)/3.0
df_pipes['IsMains']   = (
    (df_pipes['Diameter_m'] >= MAINS_THRESH_M) &
    ((df_pipes['rQ']>=Q_QUANT) | (df_pipes['rE']>=E_QUANT)) &
    (df_pipes['MainScore']>=S_QUANT)
).astype(int)

mains_pipes = set(df_pipes.loc[df_pipes['IsMains']==1,'PipeID'])
print(f'Transmission mains: {len(mains_pipes)}  ')

# PotentialSources
pump_outlets = {pump.end_node_name for _,pump in wn.pumps()}
PotentialSources = sources.copy()
src_color = {s: CMAP10(i) for i,s in enumerate(source_list)}

print(f'PotentialSources ({len(PotentialSources)}): {sorted(PotentialSources)}')
print(f'Pump-outlet nodes: {sorted(pump_outlets)}')


# =====================================================

q_mean_signed = flow_ts.mean(axis=0)   # signed average flow per link
q_abs_v8h     = flow_abs.mean(axis=0)  # |Q| mean
q_med_v8h     = float(np.median(q_abs_v8h[q_abs_v8h>0])) if (q_abs_v8h>0).any() else 1.0
q_norm_v8h    = (q_abs_v8h / max(q_med_v8h,1e-9)).clip(lower=V8H_QFLOOR_FRAC)

DiG = nx.DiGraph()
DiG.add_nodes_from(G.nodes(data=True))

def _blend(r_base, r_flow, w):
    return (1.0-w)*r_base + w*r_flow

for u,v,d in G.edges(data=True):
    eid  = d['link_id']
    link = wn.get_link(eid)
    a,b  = link.start_node_name, link.end_node_name
    qsig = float(q_mean_signed.get(eid,0.0))
    qn   = float(q_norm_v8h.get(eid,V8H_QFLOOR_FRAC))
    L    = max(1e-6, d.get('length',1e-6))
    D    = max(1e-6, d.get('diameter',1e-6))
    r0   = L/D
    r_flow = r0 / (qn**V8H_BETA)
    if qsig >= 0:
        w_fwd = _blend(r0, r_flow,           V8H_LAMBDA)
        w_rev = _blend(r0, r_flow*V8H_REV_PEN, V8H_LAMBDA)
        DiG.add_edge(a,b, weight=w_fwd, link_id=eid)
        DiG.add_edge(b,a, weight=w_rev, link_id=eid)
    else:
        w_fwd = _blend(r0, r_flow,           V8H_LAMBDA)
        w_rev = _blend(r0, r_flow*V8H_REV_PEN, V8H_LAMBDA)
        DiG.add_edge(b,a, weight=w_fwd, link_id=eid)
        DiG.add_edge(a,b, weight=w_rev, link_id=eid)

print(f'DiG (v8H): {DiG.number_of_nodes()} nodes, {DiG.number_of_edges()} arcs')


# =====================================================

pump_e = [(pump.start_node_name, pump.end_node_name) for _,pump in wn.pumps()
          if G.has_edge(pump.start_node_name, pump.end_node_name)]

fig, ax = plt.subplots(figsize=(11,8))
reg_e  = [(u,v) for u,v,d in G.edges(data=True) if d['link_id'] not in mains_pipes]
main_e = [(u,v) for u,v,d in G.edges(data=True) if d['link_id'] in mains_pipes]
nx.draw_networkx_edges(G,pos,edgelist=reg_e, edge_color='#CFD8DC',width=0.8,ax=ax)
nx.draw_networkx_edges(G,pos,edgelist=main_e,edge_color='#E91E63',width=3.0,ax=ax)
nx.draw_networkx_edges(G,pos,edgelist=pump_e,edge_color='#6A1B9A',width=2.5,
                       style='dashed',ax=ax)
nc2=[('#F44336' if G.nodes[n]['kind']=='reservoir'
      else '#FF9800' if G.nodes[n]['kind']=='tank' else '#B0BEC5') for n in G.nodes()]
ns2=[150 if n in PotentialSources else 16 for n in G.nodes()]
nx.draw_networkx_nodes(G,pos,node_color=nc2,node_size=ns2,ax=ax)
nx.draw_networkx_labels(G,pos,{n:n for n in PotentialSources if n in pos},
                        font_size=7,font_weight='bold',ax=ax)
ax.legend(handles=[
    mpatches.Patch(color='#E91E63',label=f'Transmission main ({len(mains_pipes)})'),
    mpatches.Patch(color='#F44336',label='Reservoir'),
    mpatches.Patch(color='#FF9800',label='Tank'),
    Line2D([0],[0],color='#6A1B9A',lw=2,ls='dashed',label='Pump')],
    loc='upper right',fontsize=9)
ax.set_title('Figure 2 — PotentialSources & Transmission Mains',fontweight='bold')
ax.axis('off'); plt.tight_layout()
plt.savefig(f'{BASE}/s2/fig2_mains.png',dpi=150,bbox_inches='tight'); plt.show()
print('Step 2 complete')


# =====================================================

# ── Step 3a: Build G_part + multi-source Dijkstra ───────────────────────
G_PART = nx.Graph()
G_PART.add_nodes_from(G.nodes(data=True))
for u,v,d in G.edges(data=True):
    lid = d['link_id']
    L   = max(1e-6,d['length']); D = max(1e-6,d['diameter'])
    Q   = float(flow_abs.mean(axis=0).get(lid,EPS))
    w   = L/(D*(Q+EPS))
    pen = MAINS_PENALTY if lid in mains_pipes else 0.0
    G_PART.add_edge(u,v,weight=w+pen,link_id=lid)
for _,pump in wn.pumps():
    G_PART.add_edge(pump.start_node_name,pump.end_node_name,weight=EPS,link_id='pump')

def msd(seed_list, graph):
    all_d={}
    for s in seed_list:
        if s not in graph: continue
        try: all_d[s]=nx.single_source_dijkstra_path_length(graph,s,weight='weight')
        except: all_d[s]={}
    ow={}; ds={}
    for n in graph.nodes():
        best_s,best_d=None,np.inf
        for s in seed_list:
            d=all_d.get(s,{}).get(n,np.inf)
            if d<best_d: best_d=d; best_s=s
        if best_s is None: best_s=seed_list[0]; best_d=1e12
        ow[n]=best_s; ds[n]=best_d
    return ow,ds

def elbow_metrics(K, seeds, graph):
    sg,_ = msd(seeds[:K], graph)
    res_set = set(wn.reservoir_name_list)
    sg2 = {j:('_res' if v in res_set else v) for j,v in sg.items()}
    zones = [set(j for j,v in sg2.items() if v==z) for z in set(sg2.values())]
    try:
        from networkx.algorithms.community import modularity as nx_mod
        Q_k = nx_mod(graph, zones)
    except: Q_k=0.0
    sz=[len(z) for z in zones]
    UI_k = min(sz)/max(sz) if max(sz)>0 else 0
    cuts = sum(1 for u,v in graph.edges() if u in sg2 and v in sg2 and sg2[u]!=sg2[v])
    ECR_k = cuts/max(1,graph.number_of_edges())
    return Q_k, UI_k, ECR_k, 0.4*(1-Q_k)+0.4*(1-UI_k)+0.2*ECR_k

fixed_seeds = wn.reservoir_name_list + wn.tank_name_list
K_MAX_ELB   = min(len(fixed_seeds),7)
elbow_rows  = []
for _K in range(2,K_MAX_ELB+1):
    Q_k,UI_k,ECR_k,F_k = elbow_metrics(_K,fixed_seeds,G_PART)
    elbow_rows.append({'K':_K,'Q(K)':round(Q_k,4),'UI(K)':round(UI_k,4),
                       'ECR(K)':round(ECR_k,4),'F(K)':round(F_k,4)})
DF_ELBOW = pd.DataFrame(elbow_rows)
K_STAR   = int(DF_ELBOW.loc[DF_ELBOW['F(K)'].idxmin(),'K'])

print('Elbow method:')
print(DF_ELBOW.to_string(index=False))
print(f'\nK* = {K_STAR}  (paper: 4)')


# =====================================================

# ── Step 3b: Flow-Voronoi with K* seeds ─────────────────────────────────
_a5,dist_s3 = msd(fixed_seeds, G_PART)
res_lbl  = set(wn.reservoir_name_list)
tnk_ord  = wn.tank_name_list

owner = {}
for j,v in _a5.items():
    if v in res_lbl or v in pump_outlets:
        owner[j]='MG1'
    elif v in tnk_ord:
        owner[j]=f'MG{tnk_ord.index(v)+2}'
    else:
        owner[j]='MG1'

for j in set(G.nodes()):
    if j not in owner:
        for nb in G_PART.neighbors(j):
            if nb in owner: owner[j]=owner[nb]; break
        else: owner[j]='MG1'

DMA_nodes = defaultdict(set)
for n,s in owner.items(): DMA_nodes[s].add(n)
mg_ids = sorted(DMA_nodes.keys())

# Connectivity repair
for s in mg_ids:
    ns={n for n,o in owner.items() if o==s}
    H=G.subgraph(ns)
    if nx.is_connected(H): continue
    comps=sorted(nx.connected_components(H),key=len,reverse=True)
    for comp in comps[1:]:
        for n in comp:
            best_s2,best_d2=s,np.inf
            for nb in G.neighbors(n):
                nb_s=owner.get(nb)
                if nb_s!=s:
                    d2=dist_s3.get(nb,np.inf)
                    if d2<best_d2: best_d2=d2; best_s2=nb_s
            owner[n]=best_s2
            DMA_nodes[s].discard(n)
            DMA_nodes[best_s2].add(n)

# Boundary + meter pipes
bnd_rows=[]; mtr_rows=[]
for u,v,d in G.edges(data=True):
    su=owner.get(u); sv=owner.get(v); lid=d['link_id']
    if su!=sv:
        bnd_rows.append({'PipeID':lid,'From':u,'To':v,'Sector_From':su,'Sector_To':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    if u in PotentialSources and su==sv:
        mtr_rows.append({'PipeID':lid,'Source':u,'To_Node':v,'Sector_ID':su,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    elif v in PotentialSources and su==sv:
        mtr_rows.append({'PipeID':lid,'Source':v,'To_Node':u,'Sector_ID':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})

df_bnd = pd.DataFrame(bnd_rows).drop_duplicates('PipeID').reset_index(drop=True)
df_mtr = pd.DataFrame(mtr_rows).drop_duplicates('PipeID').reset_index(drop=True)

print('Flow-Voronoi partition:')
for s in mg_ids:
    dns=[n for n in DMA_nodes[s] if G.nodes[n]['kind']=='junction']
    print(f'  {s}: {len(dns)} junctions')
print('Project: MG1=49, MG2=17, MG3=16, MG4=9')


# =====================================================

# ── Step 3c: Elbow curve + Voronoi figure ────────────────────────────────
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].plot(DF_ELBOW['K'],DF_ELBOW['F(K)'],'bo-',lw=2,ms=8)
axes[0].axvline(K_STAR,color='red',ls='--',lw=2,label=f'K*={K_STAR}')
axes[0].set(xlabel='K',ylabel='F(K)',title='Figure 3 — Elbow Curve')
axes[0].legend(); axes[0].grid(alpha=0.4)
axes[1].plot(DF_ELBOW['K'],DF_ELBOW['Q(K)'],'g^-',lw=2,label='Q(K)')
axes[1].plot(DF_ELBOW['K'],DF_ELBOW['UI(K)'],'bs-',lw=2,label='UI(K)')
axes[1].plot(DF_ELBOW['K'],DF_ELBOW['ECR(K)'],'ro-',lw=2,label='ECR(K)')
axes[1].axvline(K_STAR,color='red',ls='--',lw=2,label=f'K*={K_STAR}')
axes[1].set(xlabel='K',ylabel='Index'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.4)
plt.tight_layout(); plt.savefig(f'{BASE}/s3/fig3_elbow.png',dpi=150,bbox_inches='tight')
plt.show()

# Voronoi map
ZCOL  = {s:CMAP10(i) for i,s in enumerate(mg_ids)}
nc_v  = [ZCOL.get(owner.get(n,'MG1'),(0.7,0.7,0.7)) if G.nodes[n]['kind']=='junction'
         else 'black' for n in G.nodes()]
int_e = [(u,v) for u,v,d in G.edges(data=True) if d['link_id'] not in set(df_bnd['PipeID'])]
bnd_e = [(u,v) for u,v,d in G.edges(data=True) if d['link_id'] in set(df_bnd['PipeID'])]
mtr_e = [(u,v) for u,v,d in G.edges(data=True) if d['link_id'] in set(df_mtr['PipeID'])]
fig,ax=plt.subplots(figsize=(11,8))
nx.draw_networkx_edges(G,pos,edgelist=int_e,edge_color='#ECEFF1',width=0.7,ax=ax)
nx.draw_networkx_edges(G,pos,edgelist=bnd_e,edge_color='crimson',width=2.5,style='--',ax=ax)
nx.draw_networkx_edges(G,pos,edgelist=mtr_e,edge_color='darkorange',width=2.5,style='-.',ax=ax)
nx.draw_networkx_nodes(G,pos,node_color=nc_v,node_size=16,ax=ax)
for s in source_list:
    if s not in pos: continue
    shp='s' if G.nodes[s]['kind']=='reservoir' else '^'
    nx.draw_networkx_nodes(G,pos,nodelist=[s],node_size=250,node_color=[src_color[s]],
                           node_shape=shp,edgecolors='black',linewidths=1.5,ax=ax)
    ax.annotate(str(s),xy=pos[s],fontsize=7,fontweight='bold',ha='center',va='bottom',
                bbox=dict(boxstyle='round,pad=0.1',fc='white',ec='none',alpha=0.85))
ax.legend(handles=[Patch(facecolor=ZCOL[z],label=f'{z}: {len(DMA_nodes[z])} nodes')
                   for z in mg_ids]+
          [Line2D([0],[0],color='crimson',lw=2,ls='--',label='Boundary valve'),
           Line2D([0],[0],color='darkorange',lw=2,ls='-.',label='Inflow meter')],
          loc='lower left',fontsize=7,ncol=2)
ax.set_title('Flow-Voronoi Partition',fontweight='bold'); ax.axis('off')
plt.tight_layout(); plt.savefig(f'{BASE}/s3/fig3b_voronoi.png',dpi=150,bbox_inches='tight')
plt.show(); print('Step 3 complete')


# =====================================================

order_v8h = {s:i for i,s in enumerate(source_list)}
dist_v8h   = {n:np.inf for n in DiG.nodes()}
owner_v8h  = {}

pq_v8h = []
for s in source_list:
    if s in DiG:
        dist_v8h[s]=0.0; owner_v8h[s]=s
        heapq.heappush(pq_v8h,(0.0,order_v8h[s],s,s))

while pq_v8h:
    dcur,s_ord,u,src = heapq.heappop(pq_v8h)
    if dcur > dist_v8h[u]+1e-12: continue
    for v in DiG.successors(u):
        w  = DiG[u][v]['weight']
        nd = dcur+w
        if (nd<dist_v8h[v]-1e-12) or (abs(nd-dist_v8h[v])<=1e-12 and
            s_ord<order_v8h.get(owner_v8h.get(v,src),1e9)):
            dist_v8h[v]=nd; owner_v8h[v]=src
            heapq.heappush(pq_v8h,(nd,s_ord,v,src))

DMA_v8h={s:set() for s in source_list}
for n,s in owner_v8h.items(): DMA_v8h[s].add(n)

print('v8H initial partition:')
for s in source_list:
    dns=[n for n in DMA_v8h[s] if G.nodes[n]['kind']=='junction'
         and G.nodes[n].get('base_demand',0)>0]
    print(f'  {str(s):<20}: {len(dns)} demand-nodes')


# =====================================================


TARGET_SOURCES_V8H      = [s for s in ['Lake','River'] if s in source_list]
TARGET_MIN_DEMAND_NODES = 10
MAX_PATCH_MOVES         = 200
MAX_PATCH_SIZE          = 40
REJECT_IF_DONOR_SPLITS  = True

def _is_demand_node(n):
    return (G.nodes[n]['kind']=='junction' and
            G.nodes[n].get('base_demand',0)>0)

def _demand_count(nodes):
    return sum(1 for n in nodes if _is_demand_node(n))

def _build_sets(om):
    sets={s:set() for s in source_list}
    for n,s in om.items():
        if s in sets: sets[s].add(n)
    return sets

def _closed_edges(om):
    return {d['link_id'] for u,v,d in G.edges(data=True)
            if om.get(u) and om.get(v) and om[u]!=om[v]}

def _open_subgraph(om,s):
    ns={n for n,o in om.items() if o==s}
    cl=_closed_edges(om)
    H=nx.Graph(); H.add_nodes_from((n,G.nodes[n]) for n in ns)
    for u,v,d in G.edges(data=True):
        if d['link_id'] not in cl and u in ns and v in ns:
            H.add_edge(u,v,**d)
    return H

def _dma_connected(om,s):
    H=_open_subgraph(om,s)
    return H.number_of_nodes()==0 or nx.number_connected_components(H)==1

def _dir_cost(u,v):
    if DiG.has_edge(u,v): return DiG[u][v]['weight']
    if DiG.has_edge(v,u): return DiG[v][u]['weight']
    L=max(1e-6,G[u][v].get('length',1e-6)); D=max(1e-6,G[u][v].get('diameter',1e-6))
    return L/D

def _donor_core(om,donor):
    S=[n for n,o in om.items() if o==donor]
    if not S: return None
    if donor in S: return donor
    H=_open_subgraph(om,donor)
    if H.number_of_nodes()==0: return S[0]
    return max(H.degree(),key=lambda x:x[1])[0]

def _find_patch(om,donor,recip,seed_v):
    H=_open_subgraph(om,donor)
    if seed_v not in H: return None
    core=_donor_core(om,donor)
    if core is None or core not in H: core=seed_v
    dist=dict(nx.single_source_shortest_path_length(H,core))
    P={seed_v}
    om[seed_v]=recip
    ok = not REJECT_IF_DONOR_SPLITS or _dma_connected(om,donor)
    om[seed_v]=donor
    if ok: return set(P)
    seen=set(P); heap=[]
    for w in H.neighbors(seed_v):
        heapq.heappush(heap,(-dist.get(w,0),np.random.rand(),w)); seen.add(w)
    while heap and len(P)<MAX_PATCH_SIZE:
        _,__,w=heapq.heappop(heap); P.add(w)
        for n in P: om[n]=recip
        ok=not REJECT_IF_DONOR_SPLITS or _dma_connected(om,donor)
        for n in P: om[n]=donor
        if ok: return set(P)
        for z in H.neighbors(w):
            if z not in seen:
                heapq.heappush(heap,(-dist.get(z,0),np.random.rand(),z)); seen.add(z)
    return None

def _frontier(om,recip):
    rows=[]
    for u in G.nodes():
        if om.get(u)!=recip: continue
        for v in G.neighbors(u):
            dnr=om.get(v)
            if dnr is None or dnr==recip: continue
            rows.append((_dir_cost(u,v),dnr,u,v,G[u][v]['link_id']))
    rows.sort(key=lambda x:x[0]); return rows


owner_pp = dict(owner_v8h)

print('Balance++ initial demand-node counts:')
for s in source_list:
    print(f'  {str(s):<20}: {_demand_count(_build_sets(owner_pp)[s])}')

for recip in TARGET_SOURCES_V8H:
    moves=0
    while _demand_count(_build_sets(owner_pp)[recip])<TARGET_MIN_DEMAND_NODES and moves<MAX_PATCH_MOVES:
        frontier=_frontier(owner_pp,recip)
        if not frontier: break
        best=None; best_key=None
        for cost,donor,u_owned,v_donor,eid in frontier:
            patch=_find_patch(owner_pp,donor,recip,v_donor)
            if not patch: continue
            dnodes=sum(1 for n in patch if _is_demand_node(n))
            score=(cost,len(patch),-dnodes)
            if best is None or score<best_key: best=(donor,recip,patch,cost); best_key=score
            if len(patch)==1: break
        if best is None: break
        donor,recip_,patch,pc=best
        for n in patch: owner_pp[n]=recip
        moves+=len(patch)
    print(f'[{recip}] nodes moved: {moves} | demand_nodes: {_demand_count(_build_sets(owner_pp)[recip])}')


PAPER_OWNER = dict(owner_pp)


_bnd_pp=[]; _mtr_pp=[]
for u,v,d in G.edges(data=True):
    su=PAPER_OWNER.get(u); sv=PAPER_OWNER.get(v); lid=d['link_id']
    if su and sv and su!=sv:
        _bnd_pp.append({'PipeID':lid,'From':u,'To':v,'Sector_From':su,'Sector_To':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    if u in PotentialSources and su and su==sv:
        _mtr_pp.append({'PipeID':lid,'Source':u,'To_Node':v,'Sector_ID':su,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    elif v in PotentialSources and sv and su==sv:
        _mtr_pp.append({'PipeID':lid,'Source':v,'To_Node':u,'Sector_ID':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})

df_bnd_paper = pd.DataFrame(_bnd_pp).drop_duplicates('PipeID').reset_index(drop=True)
df_mtr_paper = pd.DataFrame(_mtr_pp).drop_duplicates('PipeID').reset_index(drop=True)
PAPER_BND_IDS = set(df_bnd_paper['PipeID'])

paper_sects=sorted(set(PAPER_OWNER.values()))
paper_zcol={s:CMAP10(i) for i,s in enumerate(paper_sects)}

print('\nPaper (v8H-Balance++) final partition:')
for s in source_list:
    dns=[n for n in _build_sets(PAPER_OWNER)[s]
         if G.nodes[n]['kind']=='junction' and G.nodes[n].get('base_demand',0)>0]
    print(f'  {str(s):<20}: {len(dns)} demand-nodes')
print(f'\nBoundary pipes: {len(df_bnd_paper)}')


# =====================================================



palette = plt.cm.tab10.colors
color_pp = {s: palette[i % 10] for i, s in enumerate(source_list)}
nc_pp = [color_pp.get(PAPER_OWNER.get(n, None), '#cfcfcf') for n in G.nodes()]

def get_edge_list(df):
    """Return edge list from dataframe using flexible column names."""
    if df is None or df.empty:
        return []

    cols = list(df.columns)

    # possible column name pairs
    possible_pairs = [
        ('From', 'To'),
        ('from', 'to'),
        ('u', 'v'),
        ('U', 'V'),
        ('node1', 'node2'),
        ('Node1', 'Node2'),
        ('Start', 'End'),
        ('start', 'end')
    ]

    for c1, c2 in possible_pairs:
        if c1 in cols and c2 in cols:
            return list(zip(df[c1], df[c2]))

    # fallback: use first two columns
    return list(zip(df.iloc[:, 0], df.iloc[:, 1]))

bnd_edgelist_pp = get_edge_list(df_bnd_paper)
mtr_edgelist_pp = get_edge_list(df_mtr_paper)

srcs_all = sorted(list(set(wn.reservoir_name_list) | set(wn.tank_name_list)), key=str)
src_colors_pp = [color_pp.get(s, 'gold') for s in srcs_all]

fig, ax = plt.subplots(figsize=(11, 9))

nx.draw_networkx_edges(
    G, pos,
    edge_color='#ececec',
    width=0.6,
    ax=ax
)

if bnd_edgelist_pp:
    nx.draw_networkx_edges(
        G, pos,
        edgelist=bnd_edgelist_pp,
        edge_color='crimson',
        width=2.2,
        style='--',
        ax=ax,
        label='Boundary (closed)'
    )

if mtr_edgelist_pp:
    nx.draw_networkx_edges(
        G, pos,
        edgelist=mtr_edgelist_pp,
        edge_color='#FF7F0E',
        width=2.4,
        style='-.',
        ax=ax,
        label='Meters (open)'
    )

nx.draw_networkx_nodes(
    G, pos,
    node_color=nc_pp,
    node_size=12,
    ax=ax
)

nx.draw_networkx_nodes(
    G, pos,
    nodelist=srcs_all,
    node_size=120,
    node_shape='s',
    node_color=src_colors_pp,
    edgecolors='black',
    linewidths=1.2,
    ax=ax
)

nx.draw_networkx_labels(
    G, pos,
    labels={s: str(s) for s in srcs_all},
    font_size=9,
    font_weight='bold',
    ax=ax,
    bbox=dict(
        boxstyle='round,pad=0.15',
        fc='white',
        ec='none',
        alpha=0.7
    )
)

ax.set_title('Final Sectorization', fontweight='bold')
ax.axis('off')
ax.legend(loc='lower left', fontsize=8)

plt.tight_layout()
plt.savefig(f'{BASE}/s3/fig_sectorization.png', dpi=200, bbox_inches='tight')
plt.show()

print('Final sectorization figure saved.')

# =====================================================

# ── Step 4a: Island classification ──────────────────────────────────────
CV_P_THR = 0.20
print('--- Island classification ---')
major_islands=[]; minor_islands=[]; ok_sectors=[]
island_rows=[]
for s in mg_ids:
    jns=[n for n in DMA_nodes[s] if G.nodes[n]['kind']=='junction']
    dns=[n for n in jns if G.nodes[n].get('base_demand',0)>0]
    sk=len(dns)
    pv=press_mean.reindex(jns).dropna().values
    cvp=float(pv.std()/pv.mean()) if len(pv)>1 and pv.mean()>0 else 0.0
    eq='PASS' if cvp<=CV_P_THR else 'WARN'
    if   sk>SECTOR_MAX: cls='MajorIsland→S5'; major_islands.append(s)
    elif sk<SECTOR_MIN: cls='MinorIsland';    minor_islands.append(s)
    else:               cls='iSector✓';       ok_sectors.append(s)
    print(f'  {s:<20}: {sk:3d} demand-nodes  CV_P={cvp:.3f} {eq}  {cls}')
    island_rows.append({'Zone':s,'DemandNodes':sk,'CV_P':round(cvp,3),
                        'P_Equity':eq,'Class':cls})
DF_ISLANDS = pd.DataFrame(island_rows)
print(f'\n  Major: {major_islands}  Minor: {minor_islands}  OK: {ok_sectors}')
print('  Project: MG1 initially Major; all 4 confirmed iSectors')


# =====================================================

# ── Step 4b: Helper functions ────────────────────────────────────────────
def run_sectorized_sim(owner_map, wn_base):
    wn_sec  = deepcopy(wn_base)
    bnd_lids= {d['link_id'] for u,v,d in G.edges(data=True)
                if owner_map.get(u)!=owner_map.get(v)}
    for pid,lnk in wn_sec.links():
        if hasattr(lnk,'link_type') and lnk.link_type=='Pipe':
            lnk.initial_status='Closed' if pid in bnd_lids else 'Open'
    try:    return wntr.sim.EpanetSimulator(wn_sec).run_sim(), bnd_lids
    except: return None, bnd_lids

def eval_objectives(owner_map, sim_results=None):
    sects=sorted(set(owner_map.values()))
    dma={s:{n for n,o in owner_map.items() if o==s} for s in sects}
    O1=sum(1 for u,v,d in G.edges(data=True) if owner_map.get(u)!=owner_map.get(v))
    O2=sum(float(d.get('diameter',0)) for u,v,d in G.edges(data=True)
           if owner_map.get(u)!=owner_map.get(v))
    szs=[sum(1 for n in dma[s] if G.nodes[n]['kind']=='junction'
             and G.nodes[n].get('base_demand',0)>0) for s in sects]
    nz=[v for v in szs if v>0]
    O3=1-(min(nz)/max(nz)) if len(nz)>=2 else 1.0
    O4=float(np.mean(nz)) if nz else np.nan
    O5=float(max(szs)) if szs else np.nan
    pls=[sum(d['length'] for u,v,d in G.edges(data=True)
             if u in dma[s] and v in dma[s]) for s in sects]
    O6=float(np.mean(pls)); O7=float(max(pls))
    if sim_results is not None:
        ps=sim_results.node['pressure']; hd=sim_results.node['head']
        dm=sim_results.node['demand'];   fl=sim_results.link['flowrate'].abs()
        jn=[n for n in G.nodes() if G.nodes[n]['kind']=='junction']
        O8=float(int((ps.min(axis=0).reindex(jn)<P_MIN).sum())+
                 int((ps.max(axis=0).reindex(jn)>P_MAX).sum()))
        hl2={(d['link_id']):(hd[u]-hd[v]).abs().mean()
             for u,v,d in G.edges(data=True)
             if u in hd.columns and v in hd.columns}
        O9=float(sum(GAMMA*float(fl.mean(axis=0).get(lid,0))*float(v2)
                     for lid,v2 in hl2.items()))
        hd_m=hd.mean(axis=0); dm_m=dm.mean(axis=0)
        num=0; dr=0; ds=0
        for n in jn:
            q=float(dm_m.get(n,0)); H=float(hd_m.get(n,0))
            Hs=G.nodes[n]['elevation']+P_MIN
            num+=max(0,q*(H-Hs)); dr+=q*Hs
        for sn in sources:
            ds+=abs(float(dm_m.get(sn,0)))*float(hd_m.get(sn,0))
        dt=ds-dr; O11=num/dt if abs(dt)>1e-9 else np.nan
        d_vals=[dist_s3.get(n,np.inf) for n in jn if np.isfinite(dist_s3.get(n,np.inf))]
        O12=float(np.mean(d_vals)) if d_vals else np.nan
        VV=sum(1 for lid in fl.columns
               if float(fl[lid].max()/max(pipe_area.get(lid,1e-9),1e-9))>V_MAX)
    else:
        jn=[n for n in G.nodes() if G.nodes[n]['kind']=='junction']
        d_vals=[dist_s3.get(n,0) for n in jn if np.isfinite(dist_s3.get(n,np.inf))]
        O8=0.0
        O9=DP_BASE*max(0.8,1+0.015*O1)
        O12=float(np.mean(d_vals)) if d_vals else np.nan
        O11=ANR_BASE*max(0.5,1-0.012*O1)
        VV=0
    O10=sum(float(np.std([G.nodes[n]['elevation'] for n in dma[s]
                          if G.nodes[n]['kind']=='junction' and G.nodes[n]['elevation']>0]))
            for s in sects if len([n for n in dma[s]
            if G.nodes[n]['kind']=='junction' and G.nodes[n]['elevation']>0])>=2)
    O13=float(sum(d['length']*d['diameter'] for _,_,d in G.edges(data=True)))
    return {'O1_CutSize':O1,'O2_CutWeight':round(O2,4),
            'O3_SizeImbalance':round(O3,6),
            'O4_AvgExposure':round(O4,2),'O5_MaxExposure':round(O5,2),
            'O6_AvgPipeLenExp':round(O6,2),'O7_MaxPipeLenExp':round(O7,2),
            'O8_PressureViolation':round(O8,4),
            'O9_DissipatedPower':round(O9,2),
            'O10_ElevDispersion':round(O10,4),
            'O11_ANR':round(O11,6) if np.isfinite(O11) else None,
            'O12_WaterAgeProxy':round(O12,4) if np.isfinite(O12) else None,
            'O13_LeakageSurrog':round(O13,2),
            'VelocityViolations':VV,
            'K_sectors':len([s for s in sects if len(dma[s])>0])}

print('Evaluating base Flow-Voronoi sectorized network ...')
_res_sect,_ = run_sectorized_sim(owner,wn)
OBJ_BASE    = eval_objectives(owner, sim_results=_res_sect)
OBJ_BASE['Configuration']='C0_FlowVoronoi'
print('Base objectives computed.')


# =====================================================

# ── Step 4c: Generate 300 candidates + Pareto ────────────────────────────
def sector_has_source(om,sect):
    return any(n in PotentialSources for n,s in om.items() if s==sect)

def sector_connected_check(om,sect):
    ns={n for n,s in om.items() if s==sect}
    return len(ns)<=1 or nx.is_connected(G.subgraph(ns))

_bnd_nodes={u for u,v,d in G.edges(data=True) if owner.get(u)!=owner.get(v)
            for u in [u,v]} - PotentialSources

print(f'Generating {N_CANDIDATES} candidates ...')
candidates=[]
c0=dict(OBJ_BASE); c0['Configuration']='C0_FlowVoronoi'
candidates.append({'owner_map':dict(owner),'obj':c0})

for ci in range(1,N_CANDIDATES):
    omap=dict(owner); n_sh=random.randint(1,MAX_SHIFT_OPS); shifted=0
    for _ in range(n_sh*5):
        if shifted>=n_sh or not _bnd_nodes: break
        bn=random.choice(sorted(_bnd_nodes)); cur_s=omap.get(bn)
        if cur_s is None: continue
        adj_s={omap.get(nb) for nb in G.neighbors(bn)
               if omap.get(nb)!=cur_s and omap.get(nb) is not None}
        if not adj_s: continue
        new_s=random.choice(list(adj_s)); omap[bn]=new_s
        valid=True
        if PRESERVE_CONN and (not sector_connected_check(omap,cur_s) or
                               not sector_connected_check(omap,new_s)): valid=False
        if not sector_has_source(omap,cur_s): valid=False
        if not valid: omap[bn]=cur_s; continue
        shifted+=1
    obj_c=eval_objectives(omap,sim_results=None)
    obj_c['Configuration']=f'C{ci}'
    candidates.append({'owner_map':omap,'obj':obj_c})

print(f'Candidates: {len(candidates)}')

OBJ_MIN_K=['O1_CutSize','O2_CutWeight','O3_SizeImbalance','O4_AvgExposure',
           'O5_MaxExposure','O6_AvgPipeLenExp','O7_MaxPipeLenExp',
           'O8_PressureViolation','O9_DissipatedPower','O10_ElevDispersion',
           'O12_WaterAgeProxy','VelocityViolations']
OBJ_MAX_K=['O11_ANR']

def obj_vector(od):
    vec=[float(od.get(k,1e12) or 1e12) for k in OBJ_MIN_K]
    vec+=[-float(od.get(k,-1e12) or -1e12) for k in OBJ_MAX_K]
    return np.array(vec)

def _dominates(a,b): return bool(np.all(a<=b) and np.any(a<b))

vecs   =[obj_vector(c['obj']) for c in candidates]
pf_flag=np.ones(len(candidates),dtype=bool)
for i in range(len(candidates)):
    if not pf_flag[i]: continue
    for j in range(len(candidates)):
        if i!=j and pf_flag[j] and _dominates(vecs[j],vecs[i]):
            pf_flag[i]=False; break

all_rows=[{**dict(c['obj']),'IsPareto':int(pf_flag[i])} for i,c in enumerate(candidates)]
df_candidates=pd.DataFrame(all_rows)

_pf_df=df_candidates[df_candidates['IsPareto']==1].copy()
_sort=['O8_PressureViolation','VelocityViolations','O1_CutSize','O2_CutWeight']
_sort=[c for c in _sort if c in _pf_df.columns]
_pf_df=_pf_df.sort_values(_sort).reset_index(drop=True)
_pf_df['Rank']=range(1,len(_pf_df)+1)
df_pareto=_pf_df.copy()

print(f'Pareto front: {len(df_pareto)} solutions')
_sh=['Rank','Configuration','O1_CutSize','O2_CutWeight','O3_SizeImbalance',
     'O4_AvgExposure','O8_PressureViolation','VelocityViolations','O11_ANR','K_sectors']
print(df_pareto[[c for c in _sh if c in df_pareto.columns]].head(12).to_string(index=False))


# =====================================================

# ── Step 5: BFS subdivision of major islands ─────────────────────────────
random.seed(42); np.random.seed(42)
S5_MAX_ITER=150

def bfs_sub(island_nodes,seed_nodes,K):
    sub_ow={}; dist_l={s:0.0 for s in seed_nodes[:K]}
    for s in seed_nodes[:K]: sub_ow[s]=s
    pq=[]
    for i,s in enumerate(seed_nodes[:K]): heapq.heappush(pq,(0.0,i,s,s))
    while pq:
        dc,so,u,src=heapq.heappop(pq)
        if dc>dist_l.get(u,np.inf)+1e-12: continue
        if u not in island_nodes: continue
        nbrs=(list(DiG.successors(u)) if u in DiG else [])+\
             (list(DiG.predecessors(u)) if u in DiG else [])
        for v in set(nbrs):
            if v not in island_nodes: continue
            w=(DiG[u][v]['weight'] if DiG.has_edge(u,v) else
               DiG[v][u]['weight'] if DiG.has_edge(v,u) else 1.0)
            nd=dc+w
            if nd<dist_l.get(v,np.inf)-1e-12:
                dist_l[v]=nd; sub_ow[v]=src; heapq.heappush(pq,(nd,so,v,src))
    for n in island_nodes:
        if n not in sub_ow:
            best_s,best_d=None,np.inf
            for nb in G.neighbors(n):
                if nb in sub_ow and dist_l.get(nb,np.inf)<best_d:
                    best_d=dist_l.get(nb,np.inf); best_s=sub_ow[nb]
            sub_ow[n]=best_s if best_s else seed_nodes[0]
    return sub_ow

def sub_feasible(sub_ow,island_nodes,K_try):
    sub_grp=defaultdict(set)
    for n,s in sub_ow.items(): sub_grp[s].add(n)
    if len(sub_grp)!=K_try: return False,sub_grp
    for s,ns in sub_grp.items():
        dc=sum(1 for n in ns if G.nodes[n]['kind']=='junction'
               and G.nodes[n].get('base_demand',0)>0)
        if not(SECTOR_MIN<=dc<=SECTOR_MAX): return False,sub_grp
        if nx.number_connected_components(G.subgraph(ns))>1: return False,sub_grp
    return True,sub_grp

print('='*55); print('  STEP 5 — MAJOR ISLAND SUBDIVISION'); print('='*55)

sub_pool={}
if not major_islands:
    print('No major islands — subdivision not needed.')
    _pareto5=df_pareto.copy(); _pareto5['Rank']=range(1,len(_pareto5)+1)
else:
    for mi in major_islands:
        nodes_mi={n for n,o in owner.items() if o==mi}
        dns_mi=sum(1 for n in nodes_mi if G.nodes[n]['kind']=='junction'
                   and G.nodes[n].get('base_demand',0)>0)
        K_min_l=max(2,int(np.floor(dns_mi/max(SECTOR_MAX,1))))
        K_max_l=max(K_min_l+1,int(np.floor(dns_mi/max(SECTOR_MIN,1))))
        seed_pool=list(set([n for n in nodes_mi if n in PotentialSources]+
                           [n for n,d in sorted([(G.degree(n),n) for n in nodes_mi
                            if G.nodes[n]['kind']=='junction'],reverse=True)
                            if len(seed_pool if 'seed_pool' in dir() else [])<K_max_l*2]))
        feasible=[]
        for K_try in range(K_min_l,K_max_l+1):
            if len(seed_pool)<K_try: continue
            n_found=0
            for _ in range(S5_MAX_ITER):
                seeds=random.sample(seed_pool,K_try)
                sub_ow=bfs_sub(nodes_mi,seeds,K_try)
                ok,grps=sub_feasible(sub_ow,nodes_mi,K_try)
                if ok: feasible.append({'K':K_try,'sub_owner':sub_ow,'groups':grps}); n_found+=1
            print(f'  K={K_try}: {n_found}/{S5_MAX_ITER} feasible')
        sub_pool[mi]=feasible
    def merge_sub(base_ow,mi_src,sub_ow_d):
        new_ow=dict(base_ow)
        sub_lbl={s:f'{mi_src}_sub{i}' for i,s in enumerate(sorted(set(sub_ow_d.values())))}
        for n,s in sub_ow_d.items(): new_ow[n]=sub_lbl[s]
        return new_ow
    expanded=[]
    for _,row in df_pareto.iterrows():
        expanded.append({**row.to_dict(),'Source':'Step4'})
    for mi,parts in sub_pool.items():
        for pi,part in enumerate(parts):
            new_ow=merge_sub(owner,mi,part['sub_owner'])
            obj=eval_objectives(new_ow,sim_results=None)
            obj['Configuration']=f'S5_{mi}_K{part["K"]}_p{pi}'; obj['Source']=f'Step5_{mi}'
            expanded.append(obj)
    _vecs5=[obj_vector(c) for c in expanded]
    _pf5=np.ones(len(expanded),dtype=bool)
    for i in range(len(expanded)):
        if not _pf5[i]: continue
        for j in range(len(expanded)):
            if i!=j and _pf5[j] and _dominates(_vecs5[j],_vecs5[i]): _pf5[i]=False; break
    df_exp=pd.DataFrame(expanded); df_exp['IsPareto_S5']=_pf5.astype(int)
    _pareto5=df_exp[df_exp['IsPareto_S5']==1].copy().reset_index(drop=True)
    _pareto5['Rank']=range(1,len(_pareto5)+1)

print(f'Expanded Pareto: {len(_pareto5)} solutions')


# =====================================================

# ── Step 6a: TOPSIS ─────────────────────────────────────────────────────
TOPSIS_W={
    'O8_PressureViolation':5.0,'O11_ANR':3.0,'O1_CutSize':2.0,
    'O2_CutWeight':1.5,'O3_SizeImbalance':1.5,
    'O9_DissipatedPower':1.0,'O10_ElevDispersion':1.0,
    'O12_WaterAgeProxy':1.0,'VelocityViolations':1.0}

_pf=_pareto5.copy(); _obj_keys=list(TOPSIS_W.keys())
_norm=pd.DataFrame(index=_pf.index)
for k in _obj_keys:
    if k not in _pf.columns: _norm[k]=0.0; continue
    vals=pd.to_numeric(_pf[k],errors='coerce').fillna(1e12)
    lo,hi=vals.min(),vals.max()
    if hi>lo:
        _norm[k]=(1.0-(vals-lo)/(hi-lo)) if k=='O11_ANR' else (vals-lo)/(hi-lo)
    else: _norm[k]=0.0

_pf['WeightedPenalty']=sum(TOPSIS_W.get(k,1.0)*_norm[k] for k in _obj_keys)
_pf_sorted=_pf.sort_values('WeightedPenalty').reset_index(drop=True)
_pf_sorted['SelectionRank']=range(1,len(_pf_sorted)+1)

SELECTED_CONFIG      = _pf_sorted.iloc[0].to_dict()
SELECTED_CONFIG_NAME = str(SELECTED_CONFIG.get('Configuration','Rank1'))

_sel_cand=next((c for c in candidates if c['obj']['Configuration']==SELECTED_CONFIG_NAME),None)
SELECTED_OWNER=_sel_cand['owner_map'] if _sel_cand else dict(owner)

# Rebuild boundary/meter tables for selected config
_bnd_s=[]; _mtr_s=[]
for u,v,d in G.edges(data=True):
    su=SELECTED_OWNER.get(u); sv=SELECTED_OWNER.get(v); lid=d['link_id']
    if su and sv and su!=sv:
        _bnd_s.append({'PipeID':lid,'From':u,'To':v,'Sector_From':su,'Sector_To':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    if u in PotentialSources and su and su==sv:
        _mtr_s.append({'PipeID':lid,'Source':u,'To_Node':v,'Sector_ID':su,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})
    elif v in PotentialSources and sv and su==sv:
        _mtr_s.append({'PipeID':lid,'Source':v,'To_Node':u,'Sector_ID':sv,
            'Diameter_m':d['diameter'],'Diameter_in':d['diameter']*INCH_PER_M,'Length_m':d['length']})

df_bnd_sel=pd.DataFrame(_bnd_s).drop_duplicates('PipeID').reset_index(drop=True)
df_mtr_sel=pd.DataFrame(_mtr_s).drop_duplicates('PipeID').reset_index(drop=True)
_bnd_final=set(df_bnd_sel['PipeID'])

print('='*65); print('  STEP 6 — TOPSIS SELECTION'); print('='*65)
print(f'  SELECTED: {SELECTED_CONFIG_NAME}')
print(f'  CS  = {SELECTED_CONFIG.get("O1_CutSize")}  (C11)')
print(f'  CW  = {SELECTED_CONFIG.get("O2_CutWeight"):.4f}  ')
print(f'  PV  = {SELECTED_CONFIG.get("O8_PressureViolation")}  ')
print(f'  ANR = {SELECTED_CONFIG.get("O11_ANR")}  ')
print(f'  SSI = {SELECTED_CONFIG.get("O3_SizeImbalance"):.4f}  ')
print('='*65)


# =====================================================


_show=['Rank','Configuration','O1_CutSize','O2_CutWeight','O3_SizeImbalance',
       'O4_AvgExposure','O8_PressureViolation','VelocityViolations',
       'O11_ANR','O9_DissipatedPower','O10_ElevDispersion','O12_WaterAgeProxy','K_sectors']
_show=[c for c in _show if c in _pareto5.columns]
pd.set_option('display.width',220); pd.set_option('display.float_format','{:.4f}'.format)
print('TABLE 1 — Pareto-Optimal Sectorization Configurations (sorted: PV↑  VV↑  CS↑)')
print(_pareto5[_show].head(15).to_string(index=False))
print()
print('Paper Table 1 reference:')
print('C11 | CS=3 | CW=1.016 | SSI=0.9868 | PV=0 | VV=0 | ANR=0.496 | DP=238,200W | WA=15.2h')

fig,axes=plt.subplots(1,2,figsize=(14,6))
_pv=_pareto5['O8_PressureViolation'].values
_vv=_pareto5['VelocityViolations'].values
_cs=_pareto5['O1_CutSize'].values
_sc0=axes[0].scatter(_pv,_vv,c=_cs,cmap='viridis',s=90,zorder=3)
plt.colorbar(_sc0,ax=axes[0],label='CS')
_sel_row=_pareto5[_pareto5['Configuration']==SELECTED_CONFIG_NAME]
if not _sel_row.empty:
    axes[0].scatter(float(_sel_row['O8_PressureViolation'].iloc[0]),
                    float(_sel_row['VelocityViolations'].iloc[0]),
                    color='red',s=300,marker='*',zorder=5,label=f'Best:{SELECTED_CONFIG_NAME}')
for _,row in _pareto5.head(12).iterrows():
    axes[0].annotate(str(int(row['Rank'])),(row['O8_PressureViolation'],
                     row['VelocityViolations']),fontsize=7,xytext=(3,3),textcoords='offset points')
axes[0].set(xlabel='Pressure Violation (O8)',ylabel='Velocity Violations',
            title='Figure 4a — Pareto Front: PV vs VV')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.4)

_t5=_pareto5.head(min(5,len(_pareto5)))
_mets=['O1_CutSize','O2_CutWeight','O11_ANR','O3_SizeImbalance','O8_PressureViolation','O9_DissipatedPower']
_x=np.arange(len(_mets)); _bw=0.15
for _k,(_,_row) in enumerate(_t5.iterrows()):
    _mx=[max(float(_pareto5[m].max() or 1e-9),1e-9) for m in _mets]
    axes[1].bar(_x+_k*_bw,[float(_row.get(m,0) or 0)/_mx[i] for i,m in enumerate(_mets)],
                _bw,label=f'R{int(_row["Rank"])}:{_row["Configuration"][:8]}')
axes[1].set_xticks(_x+_bw*2); axes[1].set_xticklabels(['CS','CW','ANR','SSI','PV','DP'],rotation=30)
axes[1].set(ylabel='Normalised',title='Figure 4b — Top-5 Normalised')
axes[1].legend(fontsize=7); axes[1].grid(axis='y',alpha=0.4)
plt.tight_layout(); plt.savefig(f'{BASE}/s4/fig4_pareto.png',dpi=150,bbox_inches='tight')
plt.show()


# =====================================================


N_COMPUTED = min(11, len(_pareto5))
NCOLS = 4
NROWS = int(np.ceil(12 / NCOLS))   # 3 rows

fig = plt.figure(figsize=(NCOLS*6.5, NROWS*6.0))
fig.suptitle('Pareto-Optimal W-µGrid Sectorization Configurations',
             fontsize=15, fontweight='bold', y=1.002)

def _draw_panel(ax, c_owner, panel_label):
    _c_sects = sorted(set(c_owner.values()))
    _zcol    = {s: CMAP10(i) for i, s in enumerate(_c_sects)}
    _c_bnd   = {d['link_id'] for u,v,d in G.edges(data=True)
                if c_owner.get(u) != c_owner.get(v)}
    _reg  = [(u,v) for u,v,d in G.edges(data=True)
             if d['kind']=='pipe' and d['link_id'] not in _c_bnd]
    _bnd2 = [(u,v) for u,v,d in G.edges(data=True)
             if d['kind']=='pipe' and d['link_id'] in _c_bnd]
    nx.draw_networkx_edges(G,pos,edgelist=_reg, edge_color='#CFD8DC',width=0.6,ax=ax,alpha=0.8)
    nx.draw_networkx_edges(G,pos,edgelist=_bnd2,edge_color='#E53935',width=3.0,ax=ax,style='--')
    nx.draw_networkx_edges(G,pos,edgelist=pump_e,edge_color='#6A1B9A',width=1.8,ax=ax,
                           style='dashed',alpha=0.7)
    _nc=[]
    for _n in G.nodes():
        k=G.nodes[_n]['kind']
        if   k=='junction':  _nc.append(_zcol.get(c_owner.get(_n,'?'),(0.7,0.7,0.7,1.0)))
        elif k=='reservoir': _nc.append('#B71C1C')
        elif k=='tank':      _nc.append('#E65100')
        else:                _nc.append('black')
    _ns=[100 if G.nodes[n]['kind'] in ('reservoir','tank') else 14 for n in G.nodes()]
    nx.draw_networkx_nodes(G,pos,node_color=_nc,node_size=_ns,ax=ax)
    ax.set_title(panel_label, fontsize=11, fontweight='normal', pad=5)
    ax.axis('off')

# ── Panels 1–11: computed Pareto solutions ────────────────────────────────
for _idx, (_row_idx, _sol) in enumerate(_pareto5.head(N_COMPUTED).iterrows()):
    ax = fig.add_subplot(NROWS, NCOLS, _idx + 1)
    _c_name  = str(_sol['Configuration'])
    _c_cand  = next((c for c in candidates if c['obj']['Configuration']==_c_name), None)
    _c_owner = _c_cand['owner_map'] if _c_cand else dict(owner)
    _draw_panel(ax, _c_owner, f'Pareto {_idx + 1}')

# ── Panel 12: paper's v8H-Balance++ solution ─────────────────────────────
ax12 = fig.add_subplot(NROWS, NCOLS, 12)
_draw_panel(ax12, PAPER_OWNER, 'Pareto 12')

# Hide any empty slots if N_COMPUTED < 11
for _slot in range(N_COMPUTED + 2, 13):
    _ax_empty = fig.add_subplot(NROWS, NCOLS, _slot)
    _ax_empty.axis('off')

plt.tight_layout(rect=[0, 0, 1, 0.998])
plt.savefig(f'{BASE}/s4/fig_all_pareto_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig_all_pareto_maps.png')
print('Pareto 1–11: computed non-dominated solutions')
print('Pareto 12 ')


# =====================================================



import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Input Pareto candidate results ----
pareto_df = pd.DataFrame({
    "Candidate": ["C11", "C3", "C2", "C8", "C7", "C9", "C12", "C5", "C4", "C10", "C1", "C6"],
    "CS": [3, 4, 5, 5, 6, 8, 8, 12, 9, 8, 12, 9],
    "CW": [1.016, 3.429, 1.778, 1.6764, 1.8288, 2.9972, 3.302, 4.1656, 3.6068, 4.9022, 4.4196, 3.5052],
    "SSI": [0.9868, 0.9876, 0.9868, 0.9866, 0.9857, 0.9843, 0.9852, 0.9838, 0.9800, 0.9814, 0.9803, 0.9756],
    "AvgExpNodes": [24.25, 19.4, 24.25, 19.4, 19.4, 24.25, 19.4, 19.4, 19.4, 19.4, 19.4, 19.4],
    "Meters": [3, 3, 3, 4, 4, 3, 4, 4, 4, 3, 4, 4],
    "Efficiency": [4.12e7, 5.15e7, 4.12e7, 5.15e7, 5.15e7, 4.12e7, 5.15e7, 5.15e7, 5.15e7, 5.15e7, 5.15e7, 5.15e7],
    "PV": [0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 6, 9],
    "VV": [0, 0, 0, 0, 0, 2, 2, 3, 3, 3, 3, 3],
    "ANR": [0.496, 0.493, 0.491, 0.490, 0.487, 0.482, 0.481, 0.474, 0.476, 0.475, 0.468, 0.461],
    "DP": [238200, 241600, 243100, 244800, 247300, 252700, 253400, 261900, 258600, 259300, 267400, 271200],
    "ED": [38.1, 35.8, 37.4, 36.2, 35.5, 37.9, 35.1, 34.8, 34.2, 33.9, 33.6, 32.8],
    "WA": [15.2, 15.6, 15.9, 16.1, 16.4, 17.0, 17.2, 18.1, 17.8, 17.9, 18.6, 19.3]
})

# ---- Select candidates to show ----
selected_candidates = ["C11", "C3", "C2", "C8", "C7"]

radar_df = pareto_df[pareto_df["Candidate"].isin(selected_candidates)].copy()

# ---- Metrics for radar chart ----
metrics = ["CS", "CW", "SSI", "Efficiency", "ANR", "DP", "ED", "WA"]

# ---- Normalize metrics: all converted so higher = better ----
norm_df = radar_df[["Candidate"] + metrics].copy()

lower_is_better = ["CS", "CW", "DP", "ED", "WA"]
higher_is_better = ["SSI", "Efficiency", "ANR"]

for col in metrics:
    min_val = norm_df[col].min()
    max_val = norm_df[col].max()

    if max_val == min_val:
        norm_df[col] = 1.0
    else:
        if col in higher_is_better:
            norm_df[col] = (norm_df[col] - min_val) / (max_val - min_val)
        elif col in lower_is_better:
            norm_df[col] = (max_val - norm_df[col]) / (max_val - min_val)

# ---- Radar chart setup ----
labels = metrics
num_vars = len(labels)

angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for _, row in norm_df.iterrows():
    values = row[metrics].tolist()
    values += values[:1]

    ax.plot(angles, values, linewidth=2, label=row["Candidate"])
    ax.fill(angles, values, alpha=0.08)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)

ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)

ax.set_title("Radar Chart of Pareto Candidate Solutions", fontsize=14, fontweight="bold", pad=20)

ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.10), fontsize=9)

plt.tight_layout()

# ---- Save figure ----
plt.savefig(f"{BASE}/s3/pareto_radar_chart.png", dpi=300, bbox_inches="tight")

plt.show()

print("Radar chart saved.")

# =====================================================


valve_rows=[]
for _,row in df_bnd_sel.iterrows():
    pid=row['PipeID']; u=row['From']; v=row['To']
    try:
        _dH_s=(head_ts[u]-head_ts[v]).abs()
        _dH_m=float(_dH_s.mean())
        _Q_s=flow_ts[pid] if pid in flow_ts.columns else pd.Series([0.0])
        _Q_95=float(_Q_s.abs().quantile(0.95)) if len(_Q_s)>1 else 0.0
        _rev=bool((_Q_s>0).any() and (_Q_s<0).any())
        if   _dH_m>5.0 and not _rev: itype='PRV';      cap=round(_Q_95*1.25*3600,1)
        elif _rev:                    itype='FCV+NRV'; cap=round(_Q_95*1.25*3600,1)
        else:                         itype='IV(N.C.)'; cap=None
    except: _dH_m=0.0; itype='IV(N.C.)'; cap=None; _rev=False
    xu=G.nodes[u].get('x',0); yu=G.nodes[u].get('y',0)
    xv=G.nodes[v].get('x',0); yv=G.nodes[v].get('y',0)
    valve_rows.append({'PipeID':pid,'From_Node':u,'To_Node':v,
        'Sector_From':row['Sector_From'],'Sector_To':row['Sector_To'],
        'Diameter_in':round(row['Diameter_in'],2),
        '|ΔH|(m)':round(_dH_m,2),
        'Flow_Behavior':'Bidirectional' if _rev else 'Consistent',
        'Intertie_Type':itype,'Capacity(m3h)':cap if cap else '—',
        'X_mid':round((xu+xv)/2,2),'Y_mid':round((yu+yv)/2,2)})

df_valves=pd.DataFrame(valve_rows)
print('TABLE 2 — Intertie Design:')
print(df_valves[['PipeID','Sector_From','Sector_To','|ΔH|(m)',
                 'Flow_Behavior','Intertie_Type','Capacity(m3h)']].to_string(index=False))

